In [ ]:
%matplotlib notebook

import itertools
import os
import re
from collections import defaultdict
from itertools import combinations, cycle
from math import ceil

import biom
import calour as ca
import joblib
import matplotlib as mpl
import matplotlib.patches as mpatches
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scipy
import seaborn as sns
import statsmodels as sm
from IPython import display
from calour.training import plot_cm
from calour.util import compute_prevalence
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.patches import Rectangle
from matplotlib.ticker import NullLocator
from scipy.stats import randint as sp_randint
from scipy.stats import t
from skbio import DistanceMatrix, read
from skbio.stats.composition import clr
from sklearn import preprocessing
from sklearn.discriminant_analysis import QuadraticDiscriminantAnalysis
from sklearn.ensemble import RandomForestClassifier
from sklearn.feature_selection import (
    RFECV,
    SelectFromModel,
    SelectKBest,
    VarianceThreshold,
    chi2,
    f_classif,
    mutual_info_classif,
)
from sklearn.gaussian_process import GaussianProcessClassifier
from sklearn.gaussian_process.kernels import RBF
from sklearn.metrics import accuracy_score
from sklearn.model_selection import (
    GridSearchCV,
    RandomizedSearchCV,
    RepeatedStratifiedKFold,
    StratifiedKFold,
)
from sklearn.naive_bayes import GaussianNB
from sklearn.pipeline import Pipeline
from sklearn.svm import LinearSVC, SVC

/home/xuhuan/miniconda3/envs/qiime2-2022.2-my/lib/python3.8/site-packages/sklearn/utils/validation.py:70: FutureWarning: Pass n_repeats=1, random_state=None as keyword args. From version 1.0 (renaming of 0.25) passing these as positional arguments will result in an error
  warnings.warn(f"Pass {args_msg} as keyword args. From version "
/home/xuhuan/miniconda3/envs/qiime2-2022.2-my/lib/python3.8/site-packages/sklearn/utils/validation.py:70: FutureWarning: Pass n_splits=3, n_repeats=1 as keyword args. From version 1.0 (renaming of 0.25) passing these as positional arguments will result in an error
  warnings.warn(f"Pass {args_msg} as keyword args. From version "


In [3]:

def group_dist_agg(distance, group, statistic):
    '''Compute the `statistic` of the given group of distances.

    Parameters
    ----------
    distance : skbio.DistanceMatrix or square numpy.array
    group : list of tuples
        For example, [(1, 9), (5, 9), (5, 7)]
    statistic : str or Callable
        The statistic to compute for the 2-group comparison.'mean', 'median',
        or functions that accept an array of numeric and return a single numeric

    Examples
    --------
    >>> distance = np.array([[0, 0.1, 0.2],
    ...                      [0.1, 0, 0.5],
    ...                      [0.2, 0.5, 0]])
    >>> group_dist_agg(distance, [(0, 1), (1, 2)], 'mean') == 0.3
    True
    >>> group_dist_agg(distance, [(0, 1), (1, 2), (0, 2)], 'median') == 0.2
    True
    '''
    grp = np.zeros(len(group))
    for k, pair in enumerate(group):
        grp[k] = distance[pair[0], pair[1]]
    if statistic == 'mean':
        return np.mean(grp)
    elif statistic == 'median':
        return np.median(grp)
    else:
        return statistic(grp)


def distance_permute_test(distance, group_1, group_2, statistic='mean', permutations=999, random_state=None):
    '''Statistically compare if there is difference between 2 groups of distances.

    Compute the given `statistic` between 2 groups of distances and
    statistically test it by permuting the sample labels.

    Parameters
    ----------
    distance : skbio.DistanceMatrix or square numpy.array
    group_1 :
    group_2 : list of tuples
        For example, [(1, 9), (5, 9), (5, 7)]
    statistic : str or Callable
        The statistic to compute for the 2-group comparison.'mean', 'median',
        or functions that accept an array of numeric and return a single numeric
    permuations : int
        Number of permutations
    random_state : None, int
        random seed

    Returns
    -------
    tuple
        p-value, the difference between 2 groups, the array of differences after permutations

    '''
    n = distance.shape[0]
    grp1 = group_dist_agg(distance, group_1, statistic)
    grp2 = group_dist_agg(distance, group_2, statistic)
    delta = grp2 - grp1
    rand = np.random.RandomState(random_state)
    delta_rand = np.zeros(permutations)
    # print(group_1, group_2)
    for k in range(permutations):
        if isinstance(distance, DistanceMatrix):
            ids = distance.ids
        else:
            ids = np.arange(n)
        idx_map = dict(zip(ids, rand.permutation(ids)))
        group_1_rand = [(idx_map[pair[0]], idx_map[pair[1]]) for pair in group_1]
        group_2_rand = [(idx_map[pair[0]], idx_map[pair[1]]) for pair in group_2]
        # print(idx_map, group_1_rand, group_2_rand)
        delta_rand[k] = group_dist_agg(distance, group_2_rand, statistic) - group_dist_agg(distance, group_1_rand, statistic)

    p = np.sum(delta_rand < delta) / (permutations + 1)
    return p, delta, delta_rand


In [4]:
np.random.seed(9)

In [5]:
import importlib.util
spec = importlib.util.spec_from_file_location("module.name", "../src/plot.py")
plot = importlib.util.module_from_spec(spec)
spec.loader.exec_module(plot)

# Import new plotting utility functions into global namespace
init_plot_style = plot.init_plot_style
rotate_xticklabels = plot.rotate_xticklabels
plot_topn_accuracy_curves = plot.plot_topn_accuracy_curves
draw_significance_brackets = plot.draw_significance_brackets
plot_grouped_topn_accuracy = plot.plot_grouped_topn_accuracy
plot_precomputed_topn_accuracy = plot.plot_precomputed_topn_accuracy
plot_boxplot_with_strip = plot.plot_boxplot_with_strip
plot_simulation_heatmap = plot.plot_simulation_heatmap

spec = importlib.util.spec_from_file_location("module.name", "../src/classify.py")
classify = importlib.util.module_from_spec(spec)
spec.loader.exec_module(classify)

In [ ]:
spec = importlib.util.spec_from_file_location("module.name", "../src/utils.py")
utils = importlib.util.module_from_spec(spec)
spec.loader.exec_module(utils)

# Inject shared functions and constants into global namespace
jackknife_se = utils.jackknife_se
get_significance_stars = utils.get_significance_stars
jackknife_auc_unpaired_test = utils.jackknife_auc_unpaired_test
save_fig = utils.save_fig
load_or_fit_model = utils.load_or_fit_model
RF_PARAM_GRID = utils.RF_PARAM_GRID
MAT_COLOR_DICT = utils.MAT_COLOR_DICT
make_rf_grid_search = utils.make_rf_grid_search
compute_jackknife_topn_stats = utils.compute_jackknife_topn_stats
extract_masked_fold_matrix = utils.extract_masked_fold_matrix
compute_auc_summary = utils.compute_auc_summary
compute_masked_jackknife_topn_stats = utils.compute_masked_jackknife_topn_stats
compute_jackknife_tensor = utils.compute_jackknife_tensor
compute_cached_jackknife_tensor = utils.compute_cached_jackknife_tensor
compute_jackknife_fold_tensor = utils.compute_jackknife_fold_tensor
compute_persistence_jackknife = utils.compute_persistence_jackknife
compute_cross_year_jackknife = utils.compute_cross_year_jackknife
plot_group_topn_jackknife = utils.plot_group_topn_jackknife
plot_precomputed_topn_jackknife = utils.plot_precomputed_topn_jackknife
plot_persistence_topn_jackknife = utils.plot_persistence_topn_jackknife
summarize_auc_grid = utils.summarize_auc_grid
welch_ttest_from_summary = utils.welch_ttest_from_summary
add_manual_errorbars = utils.add_manual_errorbars
annotate_pairwise_bar_significance = utils.annotate_pairwise_bar_significance

In [6]:
In [1]: %load_ext autoreload

In [2]: %autoreload 2

In [ ]:
fp = '../data/reference-hit.2026.biom'
exp_raw = ca.read_amplicon(fp, '../data/merged_mapping.txt',
                       normalize=None, min_reads = 0,
                       sample_metadata_kwargs=dict(na_values=['unknown'], dtype={'persistence_time_sec': str}))
exp_skin = exp_raw.filter_samples('sample_type', 'skin')
exp_skin = exp_skin.normalize(100)
exp_skin = exp_skin.filter_abundance(0.01)
keep_features = exp_skin.feature_metadata.index
exp = exp_raw.filter_ids(keep_features, axis=1)
exp = exp.normalize(100)

2026-05-11 13:11:16 WARNING These have metadata but do not have data - dropped (415): {'BLANK12.C10', 'H4.skin.TE.09.14.15.4p.5xtouch.glass.3', 'H10.skin.20x.9.14.17.plastic', 'BLANK4.4G', 'H4.skin.TE.09.14.15.11a.metal.3', 'BLANK13.2.2G', 'BLANK8.9F', 'BLANK.3.F3', 'H3.skin.TE.09.14.15.3p.metal.3', 'H2.skin.TE.09.14.15.1p.metal.2', 'BLANK8.12G', 'H1.skin.TE.09.14.15.3p.metal.1', 'BLANK12.C12', 'H3.skin.TE.09.14.15.1p.glass.4', 'H2.skin.TE.09.14.15.4p.5xtouch.plastic.3', 'H5.skin.TE.09.14.15.1p.metal.1', 'H45.surface.10x.9.12.17.metal.4', 'H20.surface.10x.9.12.17.metal.4', 'H2.surface.10x.TE.12.18.15.ceramic.3', 'BLANK8.6A', 'BLANK8.11B', 'H51.skin.20x.9.15.17.plastic', 'BLANK8.12', 'BLANK8.11G', 'surface.ctrl.TE.12.16.15.metal.4', 'BLANK8.3H', 'BLANK8.12E', 'BLANK12.A12', 'BLANK8.7G', 'H5.surface.10x.TE.12.18.15.ceramic.4', 'BLANK8.10D', 'BLANK.4.H7', 'H2.skin.TE.09.14.15.3p.plastic.4', 'BLANK.2.D2', 'BLANK.4.F10', 'BLANK8.10F', 'BLANK8.11H', 'BLANK8.9D', 'surface.ctrl.TE.09.14.15.1p.

In [15]:
md = pd.read_table('../data/merged_mapping.txt', index_col=0,
                   dtype={'month':str, 'day':str, 'year':str, 'sample_date':str},
                   parse_dates=['sample_date'])
md.pivot_table(values='sample_date', index='experiment', columns=['experiment_category'], aggfunc=len)

experiment_category,material_type,mixed_signal,number_touches,persistence
experiment,,,,
exp201509,384.0,NaN,NaN,NaN
exp201512,288.0,NaN,NaN,NaN
exp201603,369.0,NaN,NaN,111.0
exp201605,NaN,NaN,191.0,NaN
exp201610,NaN,192.0,NaN,NaN
exp201709,354.0,NaN,NaN,249.0
